# RAG Engine - C++ Build & Test

**IMPORTANT:**
1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Restart runtime
3. Run ALL cells in order (important!)

**This notebook tests:**
- C++ compilation with vcpkg
- ONNX Runtime GPU support
- Full RAG pipeline

**Time:** ~45-60 minutes total

In [ ]:
# Setup
import os
import subprocess

PROJECT_DIR = '/content/rag-engine'
VCPKG_DIR = '/content/vcpkg'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir('/content')
print('Setup complete')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!apt-get update -qq && apt-get install -y cmake build-essential git curl wget unzip make ninja-build libprotobuf-dev protobuf-compiler libuv1-dev g++ libomp-dev 2>&1 | tail -3
print('System packages installed')

In [ ]:
!rm -rf rag-engine
!git clone https://github.com/gumeeee/rag-engine.git 2>&1 | tail -3
!mkdir -p /content/rag-engine/build
print('Repository cloned')

In [ ]:
if not os.path.exists('/content/vcpkg'):
    print('Cloning vcpkg...')
    !git clone https://github.com/Microsoft/vcpkg.git /content/vcpkg 2>&1 | tail -3
print('Bootstrapping...')
!cd /content/vcpkg && ./bootstrap-vcpkg.sh 2>&1 | tail -3
print('vcpkg ready')

## Install C++ Dependencies

**This step takes 30-40 minutes.**
The vcpkg will compile ONNXRuntime and other dependencies.
This only happens once!

In [ ]:
print('Installing dependencies...')
print('This will take 30-40 minutes. Go grab a coffee!')

result = subprocess.run(
    'cd /content/vcpkg && ./vcpkg install faiss-cpu onnxruntime libuv protobuf spdlog nlohmann-json --triplet x64-linux',
    shell=True, capture_output=True, text=True, timeout=5400  # 90 min
)
print('\n=== INSTALL RESULT ===')
print(result.stdout[-5000:] if result.stdout else 'No output')
if result.returncode != 0:
    print('\nSTDERR:', result.stderr[-2000:] if result.stderr else '')

In [ ]:
# Check if dependencies are installed
import os

deps_dir = '/content/vcpkg/installed/x64-linux'
print('Checking installed dependencies...')
if os.path.exists(deps_dir):
    deps = os.listdir(deps_dir)
    print(f'Installed: {deps}')
else:
    print('Dependencies directory not found')

# Check specific libraries
lib_files = []
lib_dir = f'{deps_dir}/lib'
if os.path.exists(lib_dir):
    lib_files = [f for f in os.listdir(lib_dir) if f.endswith('.a') or f.endswith('.so')]
    print(f'\nLibraries: {lib_files[:10]}...')

## Generate Sample Corpus

In [ ]:
# Create corpus directory FIRST
import os
corpus_dir = '/content/rag-engine/data/corpus'
os.makedirs(corpus_dir, exist_ok=True)
print(f'Created directory: {corpus_dir}')

# Create documents
docs = {
    'transformer.txt': 'The Transformer uses self-attention for sequence processing.',
    'bert.txt': 'BERT uses bidirectional transformers for language understanding.',
    'gpt.txt': 'GPT uses next-token prediction for text generation.',
    'rag.txt': 'RAG combines LLM with vector retrieval.',
    'vector_search.txt': 'Vector search uses embeddings. FAISS enables efficient search.'
}
for name, content in docs.items():
    with open(f'{corpus_dir}/{name}', 'w') as f:
        f.write(content)
print(f'Created {len(docs)} documents')

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = list(docs.values())
embeddings = model.encode(texts, convert_to_numpy=True)

with open('/content/rag-engine/data/corpus.corpus', 'wb') as f:
    f.write(len(texts).to_bytes(4, 'little'))
    f.write(embeddings.shape[1].to_bytes(4, 'little'))
    embeddings.astype(np.float32).tofile(f)
print(f'Saved {len(texts)} embeddings')

## Build C++ Project

In [ ]:
import os
os.chdir('/content/rag-engine/build')
os.environ['VCPKG_ROOT'] = '/content/vcpkg'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/make'

print('Configuring CMake (should be fast now)...')
result = subprocess.run(
    'cmake .. -G "Unix Makefiles" -DCMAKE_BUILD_TYPE=Release '
    '-DCMAKE_TOOLCHAIN_FILE=/content/vcpkg/scripts/buildsystems/vcpkg.cmake '
    '-DVCPKG_TARGET_TRIPLET=x64-linux',
    shell=True, capture_output=True, text=True, timeout=600
)
print(result.stdout[-2000:] if result.stdout else 'No output')
if result.returncode != 0:
    print('ERROR:', result.stderr[-1000:] if result.stderr else '')

In [ ]:
print('Building...')
result = subprocess.run(
    'make -j$(nproc)',
    shell=True, capture_output=True, text=True, timeout=1800
)
print(result.stdout[-3000:] if result.stdout else 'No output')
if result.returncode == 0:
    print('\n=== BUILD SUCCEEDED ===')
else:
    print('\n=== BUILD FAILED ===')
    print(result.stderr[-2000:] if result.stderr else '')

In [ ]:
import os
binary = '/content/rag-engine/build/rag-engine'
if os.path.exists(binary):
    size = os.path.getsize(binary) / 1024 / 1024
    print(f'Binary found: {binary}')
    print(f'Size: {size:.1f} MB')
    print('\nBuild successful!')
else:
    print('Build failed - see errors above')